# 02 — Regresioni linear

        **Pyetja udhëheqëse:** Si vlerësojmë një parametër fizik nga matje me zhurmë?

        Ky notebook është ndërtuar si një mësim i plotë kompjuterik. Rruga kryesore ndjek:
        **problem konkret → model matematikor → zbatim nga parimet e para → bibliotekë profesionale → krahasim → interpretim**.

        ## Objektivat
        - Të nxirret zgjidhja e katrorëve më të vegjël.
- Të interpretohen residualet dhe pasiguria.
- Të krahasohet zbatimi manual me `LinearRegression`.
- Të kuptohet kondicionimi numerik.

        > **Rruga bazë për orën mësimore (45–60 min)** shënohet me seksionet **CORE**.  
        > Seksionet **EXTENDED** dhe **PLAYGROUND** janë për thellim dhe punë të pavarur.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import time

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

print("02 — Regresioni linear")
print("Seed:", RANDOM_STATE)

# 1. Ligji i Hooke-it si problem regresioni — CORE

Për një sustë ideale,
$$
F=kx.
$$
Në matje reale kemi $F_i=kx_i+\epsilon_i$. Në formë matricore, për modelin e përgjithshëm linear,
$$
\mathbf{y}=X\boldsymbol{\beta}+\boldsymbol{\epsilon}.
$$
Zgjedhim $\hat{\beta}$ duke minimizuar
$$
S(\beta)=\|\mathbf{y}-X\beta\|_2^2.
$$
Nga $\nabla_\beta S=0$ merret
$$
X^T X\hat\beta=X^T\mathbf y,
$$
dhe, kur matrica është e invertueshme,
$$
\hat\beta=(X^TX)^{-1}X^T\mathbf y.
$$
Në praktikë preferojmë zgjidhës numerikisht më të qëndrueshëm sesa inversin e drejtpërdrejtë.

In [ ]:
x = np.linspace(0.01, 0.12, 28)              # m
k_true = 42.0                                 # N/m
F = k_true*x + rng.normal(0, 0.12, len(x))    # N

fig, ax = plt.subplots()
ax.scatter(x, F)
ax.set_xlabel("Zgjatja x [m]")
ax.set_ylabel("Forca F [N]")
ax.set_title("Matje të ligjit të Hooke-it")
plt.show()

# 2. Zbatim nga parimet e para — CORE

In [ ]:
A = x[:, None]  # pa intercept, sepse fizikisht F(0)=0
k_hat, *_ = np.linalg.lstsq(A, F, rcond=None)
k_hat = float(k_hat[0])
F_hat = k_hat*x
k_hat

### Zbatimi kanonik në `src/`
Për të mos e kopjuar algoritmin në çdo notebook, kursi ruan edhe një zbatim të ripërdorshëm.
Kjo nuk e zëvendëson derivimin e mësipërm; vetëm e paketizon atë në një API të pastër.

In [ ]:
import sys
sys.path.insert(0, str(Path.cwd()))
from src.regression.linear import LinearRegressionOLS
ols_src = LinearRegressionOLS(fit_intercept=False).fit(x, F)
print("src:", ols_src.coef_[0], "manual:", k_hat)

In [ ]:
fig, ax = plt.subplots()
ax.scatter(x, F, label="matje")
ax.plot(x, F_hat, label=f"përshtatja: k={k_hat:.2f} N/m")
ax.set_xlabel("x [m]"); ax.set_ylabel("F [N]")
ax.legend(); ax.set_title("Regresioni linear për vlerësimin e k")
plt.show()

**Interpretim.** Pjerrësia e vijës është vetë parametri fizik $k$. Regresioni linear këtu nuk është thjesht një algoritëm parashikimi: është një procedurë vlerësimi parametrash.

# 3. Residualet dhe cilësia e modelit — CORE

In [ ]:
res = F - F_hat
fig, ax = plt.subplots()
ax.scatter(x, res)
ax.axhline(0, linestyle="--")
ax.set_xlabel("x [m]"); ax.set_ylabel("Residual F - F_hat [N]")
ax.set_title("Residualet")
plt.show()

In [ ]:
fig, ax = plt.subplots()
ax.hist(res, bins=10)
ax.set_xlabel("Residual [N]"); ax.set_ylabel("Frekuenca")
ax.set_title("Shpërndarja e residualeve")
plt.show()

**Interpretim.** Residualet duhet të jenë pa strukturë sistematike nëse modeli linear është i përshtatshëm. Një lakim i qartë do të sinjalizonte fizikë të munguar, jo thjesht 'gabim të modelit'.

# 4. Pasiguria e parametrit — CORE

Në modelin me një parametër dhe zhurmë homoskedastike, një vlerësim i variancës së zhurmës është
$$
s^2=\frac{\sum_i r_i^2}{n-p},
$$
ndërsa matrica e kovariancës së parametrave është
$$
\mathrm{Cov}(\hat\beta)=s^2(X^TX)^{-1}.
$$

In [ ]:
n, p = len(F), 1
s2 = np.sum(res**2)/(n-p)
cov_beta = s2*np.linalg.inv(A.T@A)
se_k = np.sqrt(cov_beta[0,0])
print(f"k = {k_hat:.3f} ± {se_k:.3f} N/m (1σ)")

# 5. Krahasimi me scikit-learn — CORE

In [ ]:
from sklearn.linear_model import LinearRegression
model = LinearRegression(fit_intercept=False).fit(x[:,None], F)
print(model.coef_[0], k_hat, model.coef_[0]-k_hat)

In [ ]:
noise_levels = np.linspace(0.02, 0.5, 20)
errors = []
for sigma in noise_levels:
    vals=[]
    for _ in range(80):
        Fsim = k_true*x + rng.normal(0, sigma, len(x))
        kval = np.linalg.lstsq(A, Fsim, rcond=None)[0][0]
        vals.append(kval-k_true)
    errors.append(np.std(vals))
fig, ax = plt.subplots()
ax.plot(noise_levels, errors, marker="o")
ax.set_xlabel("Devijimi standard i zhurmës [N]")
ax.set_ylabel("Devijimi standard i gabimit në k [N/m]")
ax.set_title("Si ndikon zhurma te pasiguria e parametrit")
plt.show()

---
## Deri këtu: materiali bazë për orën mësimore

Në një seancë rreth 60 minuta, zakonisht ndalemi këtu. Seksionet që vijojnë janë për thellim,
ushtrime të pavarura dhe eksperimentim.

# 6. Shembull i dytë: rënia e lirë — EXTENDED

Nga $s(t)=s_0+v_0t+\frac12gt^2$, nëse $s_0$ dhe $v_0$ njihen, vlerësimi i $g$ bëhet përsëri
regresion linear ndaj veçorisë $t^2/2$.

In [ ]:
t = np.linspace(0, 1.2, 30)
g_true = 9.81
s = 0.5*g_true*t**2 + rng.normal(0, 0.03, len(t))
phi = 0.5*t**2
g_hat = np.linalg.lstsq(phi[:,None], s, rcond=None)[0][0]

fig, ax = plt.subplots()
ax.scatter(t, s, label="matje")
ax.plot(t, 0.5*g_hat*t**2, label=f"g={g_hat:.3f} m/s²")
ax.set_xlabel("t [s]"); ax.set_ylabel("s [m]")
ax.legend(); ax.set_title("Vlerësimi i g nga rënia e lirë")
plt.show()

# 7. Kondicionimi numerik — EXTENDED

In [ ]:
# Dy veçori pothuajse kolineare
u = np.linspace(0, 1, 80)
Xc = np.c_[u, u + 1e-5*rng.normal(size=len(u))]
cond = np.linalg.cond(Xc.T@Xc)
cond

**Interpretim.** Kur veçoritë janë pothuajse linearisht të varura, $X^TX$ bëhet keq e kondicionuar dhe koeficientët mund të jenë shumë të paqëndrueshëm edhe kur parashikimet duken të mira.

# Ushtrime — PLAYGROUND

        ## A. Ushtrime të shkurtra
        1. Shtoni intercept dhe diskutoni kuptimin fizik të tij.
2. Dyfishoni numrin e matjeve dhe vlerësoni si ndryshon $\sigma_k$.
3. Shtoni një outlier dhe krahasojeni $k$ me rastin pa outlier.
4. Përdorni bootstrap për të vlerësuar shpërndarjen e $k$.

        ## B. Ushtrim i plotë
        Simuloni 100 eksperimente Hooke me zhurmë të ndryshme dhe verifikoni numerikisht se pasiguria e vlerësimit bie afërsisht me rritjen e numrit të pikave.

        ## C. Playground
        Ndryshoni diapazonin e zgjatjeve, zhurmën dhe numrin e matjeve.

In [ ]:
# TODO / PLAYGROUND
# Ndryshoni parametrat, shtoni grafikë ose krahasoni modele të tjera.